In [6]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Ensure project root is importable when running from /notebooks
ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.autoencoder import Autoencoder

# Load dataset
df = pd.read_csv(ROOT / "data" / "Mall_Customers.csv")

# Select features
X = df[['Annual Income (k$)', 'Spending Score (1-100)']]

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler
joblib.dump(scaler, ROOT / "models" / "scaler.pkl")

# Convert to tensor
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

# Initialize model
model = Autoencoder(input_dim=X_tensor.shape[1])

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 100

for epoch in range(epochs):
    optimizer.zero_grad()
    
    output = model(X_tensor)
    loss = criterion(output, X_tensor)
    
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# Save model
torch.save(model.state_dict(), ROOT / "models" / "autoencoder.pth")

# Extract encoded features
encoded = model.encoder(X_tensor).detach().numpy()

# Apply KMeans
kmeans = KMeans(n_clusters=5, random_state=42)
labels = kmeans.fit_predict(encoded)

# Save KMeans model
joblib.dump(kmeans, ROOT / "models" / "kmeans_autoencoder.pkl")

print("Training Complete")

Epoch 0, Loss: 1.0121455192565918
Epoch 10, Loss: 0.9089611172676086
Epoch 20, Loss: 0.7370797991752625
Epoch 30, Loss: 0.4875766336917877
Epoch 40, Loss: 0.2448863983154297
Epoch 50, Loss: 0.057016290724277496
Epoch 60, Loss: 0.03626638278365135
Epoch 70, Loss: 0.009755847044289112
Epoch 80, Loss: 0.00879561435431242
Epoch 90, Loss: 0.003720333566889167
Training Complete
